In [1]:
import ccxt
import pandas as pd
import ta
import numpy as np
import vectorbt as vbt
import warnings
warnings.filterwarnings("ignore")
import time

In [2]:
# -------------------------------------------------------------------------
# 1. 1개월 샘플 1분봉 데이터 생성 (실제 환경에서는 ccxt/거래소 API 데이터로 교체)
# -------------------------------------------------------------------------
# 1개월 = 약 30일 * 24시간 * 60분 = 43,200개 캔들
np.random.seed(42)
periods = 43200
timestamps = pd.date_range("2026-01-01", periods=periods, freq="1min")

returns = np.random.normal(loc=0.00001, scale=0.0012, size=periods)
price = 50000 * np.cumprod(1 + returns)

high = price * (1 + np.abs(np.random.normal(0, 0.0008, periods)))
low = price * (1 - np.abs(np.random.normal(0, 0.0008, periods)))
volume = np.random.lognormal(mean=2.0, sigma=0.8, size=periods)

df = pd.DataFrame(
    {"open": price, "high": high, "low": low, "close": price, "volume": volume},
    index=timestamps,
)

In [3]:
# upbit/bithumb 거래소에서 특정 기간 동안의 XRP/KRW의 1분봉 데이터를 가져오는 함수 정의
def fetch_1m_ohlcv(exchange='upbit', symbol='XRP/KRW', timeframe='1m', start_str='2026-11-01 00:00:00', end_str='2026-11-14 15:59:59'):
    exchange = ccxt.upbit({'enableRateLimit': True}) if exchange=='upbit' else ccxt.bithumb({'enableRateLimit': True})
    
    start_ts = int(pd.Timestamp(start_str, tz='Asia/Seoul').timestamp() * 1000)
    end_ts = int(pd.Timestamp(end_str, tz='Asia/Seoul').timestamp() * 1000)
    
    timeframe_minutes = 1
    timeframe_ms = timeframe_minutes * 60 * 1000
    candle_count = ((end_ts - start_ts) // timeframe_ms) + 1
    
    all_candles = []
    to_ts = end_ts
    
    print(f"[{symbol}] 분봉 데이터 수집 시작 from {start_str} to {end_str}. candle_count: {candle_count} ea.")    

    while True:
        params = {'to': exchange.iso8601(to_ts)}
        try:
            candles = exchange.fetch_ohlcv(symbol=symbol, timeframe=timeframe, limit=200, params=params)
        except Exception as e:
            print(f"Error fetching data: {e}. Retrying in 2 seconds...")
            time.sleep(2)
            continue
        
        if not candles:
            break
        
        all_candles = candles + all_candles  # Prepend new candles to the list
        oldest_ts = candles[0][0]  # 첫번째 캔들의 타임스탬프를 가져옵니다.
        
        if oldest_ts <= start_ts:
            break
        
        to_ts = oldest_ts - 1  # Move to the next batch of candles
        time.sleep(0.2)  # Sleep to respect rate limits
        
    seen = set()
    deduped = []
    for c in all_candles:
        if c[0] not in seen and start_ts <= c[0] <= end_ts:
            seen.add(c[0])
            deduped.append(c)
            
    deduped.sort(key=lambda x: x[0])  # Sort by timestamp
    
    df = pd.DataFrame(deduped, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])
    df['datetime'] = pd.to_datetime(df['timestamp'], unit='ms').dt.tz_localize('UTC').dt.tz_convert('Asia/Seoul')
    df.set_index('datetime', inplace=True)
    df.drop(columns=['timestamp'], inplace=True)
    return df


In [4]:
df_upb_1xrp = fetch_1m_ohlcv(exchange='upbit', 
                            symbol='XRP/KRW', 
                            timeframe='1m',
                            start_str='2026-11-01 00:00:00',
                            end_str='2026-11-14 15:59:59')

print(f"Fetched {len(df_upb_1xrp)} rows of data for XRP/KRW from Upbit.")
print(df_upb_1xrp.head())
print(df_upb_1xrp.tail())


[XRP/KRW] 분봉 데이터 수집 시작 from 2026-11-01 00:00:00 to 2026-11-14 15:59:59. candle_count: 19680 ea.
Fetched 19381 rows of data for XRP/KRW from Upbit.
                             open    high     low   close        volume
datetime                                                               
2026-11-01 00:00:00+09:00  1681.0  1682.0  1681.0  1681.0   8965.413178
2026-11-01 00:01:00+09:00  1681.0  1682.0  1681.0  1681.0   1308.160850
2026-11-01 00:02:00+09:00  1681.0  1682.0  1681.0  1682.0  12294.379176
2026-11-01 00:03:00+09:00  1682.0  1682.0  1681.0  1681.0   2314.271095
2026-11-01 00:04:00+09:00  1681.0  1682.0  1681.0  1681.0  17162.427640
                             open    high     low   close        volume
datetime                                                               
2026-11-14 15:55:00+09:00  1361.0  1361.0  1360.0  1361.0  29610.759792
2026-11-14 15:56:00+09:00  1361.0  1361.0  1360.0  1360.0  11456.571328
2026-11-14 15:57:00+09:00  1361.0  1361.0  1360.0  1361.0  25

In [5]:
# [2] ta 라이브러리를 사용하여 이동평균과 RSI 지표를 계산하고, 매수/매도 신호를 생성합니다. by claude

# 추세 필터
df_upb_1xrp['ema20'] = ta.trend.EMAIndicator(close=df_upb_1xrp['close'], window=20).ema_indicator()
df_upb_1xrp['ema60'] = ta.trend.EMAIndicator(close=df_upb_1xrp['close'], window=60).ema_indicator()
df_upb_1xrp['ema200'] = ta.trend.EMAIndicator(close=df_upb_1xrp['close'], window=200).ema_indicator()

# 거래량 필터
df_upb_1xrp['vol_ma_20'] = ta.trend.SMAIndicator(close=df_upb_1xrp['volume'], window=20).sma_indicator()

# 모멘텀
df_upb_1xrp['rsi'] = ta.momentum.RSIIndicator(close=df_upb_1xrp['close'], window=14).rsi()

# MACD (12, 26, 9)
macd = ta.trend.MACD(close=df_upb_1xrp['close'], window_slow=26, window_fast=12, window_sign=9)
df_upb_1xrp['macd'], df_upb_1xrp['macd_signal'], df_upb_1xrp['macd_diff'] = macd.macd(), macd.macd_signal(), macd.macd_diff()

# 변동성
atr = ta.volatility.AverageTrueRange(high=df_upb_1xrp['high'], low=df_upb_1xrp['low'], close=df_upb_1xrp['close'], window=14).average_true_range()


In [6]:
# -------------------------------------------------------------------------
# 3. 매수 / 매도 조건 생성 (Repainting 방지를 위해 shift(1) 적용)
# -------------------------------------------------------------------------
# 완성봉 기준으로 shift 처리
c_close = df_upb_1xrp["close"].shift(1)
c_open = df_upb_1xrp["open"].shift(1)
c_ema_20 = df_upb_1xrp["ema20"].shift(1)
c_ema_60 = df_upb_1xrp["ema60"].shift(1)
c_ema_200 = df_upb_1xrp["ema200"].shift(1)
c_volume = df_upb_1xrp["volume"].shift(1)
c_vol_ma = df_upb_1xrp["vol_ma_20"].shift(1)
c_rsi = df_upb_1xrp["rsi"].shift(1)
c_rsi_prev = df_upb_1xrp["rsi"].shift(2)
c_macd = df_upb_1xrp["macd"].shift(1)
c_macd_sig = df_upb_1xrp["macd_signal"].shift(1)
c_macd_prev = df_upb_1xrp["macd"].shift(2)
c_macd_sig_prev = df_upb_1xrp["macd_signal"].shift(2)

# [조건 A] 추세 필터: 종가가 EMA 60선과 200선 위에 위치
cond_trend = (c_close > c_ema_60) & (c_close > c_ema_200)

# [조건 B] 거래량 필터: 직전 봉 거래량이 20선 MA의 1.5배 이상
cond_volume = c_volume > (c_vol_ma * 1.5)

# [조건 C-1] 눌림목 반등: 단기 EMA 20선 지지 + 양봉 마감 + RSI 40~55 구간에서 상승
cond_pullback = (
    (c_close >= c_ema_20)
    & (c_close > c_open)
    & (c_rsi >= 40)
    & (c_rsi <= 55)
    & (c_rsi > c_rsi_prev)
)

# [조건 C-2] MACD 골든크로스
cond_macd_cross = (c_macd_prev <= c_macd_sig_prev) & (c_macd > c_macd_sig)

# 최종 매수 진입 신호: 추세 + 거래량 + (눌림목 반등 OR MACD 크로스)
entries = cond_trend & cond_volume & (cond_pullback | cond_macd_cross)

# 기본 매도 청산 신호: RSI 과매수(70 이상) 도달 또는 EMA 20선 이탈
# (※ 세부 익절/손절 비율은 Portfolio.from_signals 매개변수로 함께 처리)
exits = (c_rsi >= 70) | (c_close < c_ema_20)

In [7]:
# =============================================================================
# 4. 백테스트
# =============================================================================
portfolio = vbt.Portfolio.from_signals(
    close=df_upb_1xrp["close"],
    entries=entries,
    exits=exits,
    init_cash=100_000,           # 초기 자본 ($10,000)
    fees=0.0005,                # 편도 수수료 0.05%
    slippage=0.0002,            # 슬리피지 0.02%
    sl_stop=0.008,              # 고정 손절선: -0.8% 도달 시 강제 청산
    tp_stop=0.015,              # 고정 익절선: +1.5% 도달 시 강제 청산
    freq="1min",                  # 1분 주기
    accumulate=False,           # 신호 중복 매수 비활성화 (단일 포지션 유지)
)


portfolio.stats()

Start                         2026-11-01 00:00:00+09:00
End                           2026-11-14 15:59:00+09:00
Period                                 13 days 11:01:00
Start Value                                    100000.0
End Value                                    80741.2892
Total Return [%]                             -19.258711
Benchmark Return [%]                         -19.036288
Max Gross Exposure [%]                            100.0
Total Fees Paid                            12380.271018
Max Drawdown [%]                              19.319174
Max Drawdown Duration                  13 days 03:58:00
Total Trades                                        137
Total Closed Trades                                 137
Total Open Trades                                     0
Open Trade PnL                                      0.0
Win Rate [%]                                  11.678832
Best Trade [%]                                   0.7919
Worst Trade [%]                               -0

In [8]:
portfolio.plot().show()

In [14]:
portfolio.trades.records_readable

,Exit Trade Id,Column,Size,Entry Timestamp,Avg Entry Price,Entry Fees,Exit Timestamp,Avg Exit Price,Exit Fees,PnL,Return,Direction,Status,Position Id
0,0,0,59.165210,2026-11-01 07:03:00+09:00,1689.3378,49.975012,2026-11-01 07:05:00+09:00,1690.6618,50.014180,-21.654455,-0.000217,Long,Closed,0
1,1,0,59.292819,2026-11-01 08:35:00+09:00,1685.3370,49.964191,2026-11-01 08:36:00+09:00,1684.6630,49.944209,-139.871760,-0.001400,Long,Closed,1
2,2,0,59.209867,2026-11-01 08:58:00+09:00,1685.3370,49.894290,2026-11-01 08:59:00+09:00,1684.6630,49.874336,-139.676076,-0.001400,Long,Closed,2
3,3,0,59.056934,2026-11-01 09:27:00+09:00,1687.3374,49.824487,2026-11-01 09:29:00+09:00,1685.6628,49.775038,-198.496266,-0.001992,Long,Closed,3
4,4,0,58.834728,2026-11-01 10:35:00+09:00,1690.3380,49.725288,2026-11-01 10:45:00+09:00,1686.6626,49.617167,-315.583614,-0.003173,Long,Closed,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86,86,0,55.284223,2026-11-08 08:16:00+09:00,1586.3172,43.849157,2026-11-08 08:19:00+09:00,1583.6832,43.776348,-233.244149,-0.002660,Long,Closed,86
87,87,0,55.102519,2026-11-08 10:10:00+09:00,1587.3174,43.732593,2026-11-08 10:12:00+09:00,1585.6828,43.687558,-177.490728,-0.002029,Long,Closed,87
88,88,0,55.025429,2026-11-08 10:25:00+09:00,1586.3172,43.643892,2026-11-08 10:28:00+09:00,1583.6832,43.571424,-232.152297,-0.002660,Long,Closed,88
89,89,0,54.879156,2026-11-08 14:54:00+09:00,1586.3172,43.527874,2026-11-08 14:57:00+09:00,1585.6828,43.510467,-121.853677,-0.001400,Long,Closed,89
